# Phase 11: Walk-Forward Stability & Diagnostic Validation

**Scientific question**: Is EXP-019's advantage over persistence consistent, material, and robust across different forecast origins and atmospheric regimes?

**Design**:
- 4 walk-forward folds (forecast-origin windows), each retrained from scratch
- 4 models: EXP-019, EXP-017, Ridge v2, Naive Persistence
- 16 total evaluations
- 5 diagnostic dimensions: overall, seasonal, horizon-group, extreme-event, distribution-shift
- Embargo invariant verified: `val_start > train_end + 72h` for all folds
- Final test set (`X_test_v2 / y_test_v2`) never loaded

| Fold | Train Period | Val Window | Primary Seasons |
|:-----|:------------|:-----------|:----------------|
| 1 | Nov 2020 – Oct 2021 | Nov 2021 – Jan 2022 | winter_smog |
| 2 | Nov 2020 – Mar 2022 | Apr – Jun 2022 | transition, summer |
| 3 | Nov 2020 – Jun 2022 | Jul – Sep 2022 | monsoon |
| 4 | Nov 2020 – Oct 2023 | Nov 2023 – Jan 2024 | winter_smog (2-yr gap) |

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

with open('../data/models/walk_forward/walk_forward_report.json') as f:
    report = json.load(f)

df = pd.read_csv('../data/models/walk_forward/walk_forward_summary.csv')

MODEL_LABELS = {
    'exp019': 'EXP-019 (Pers. Hybrid)',
    'exp017': 'EXP-017 (Hybrid Specialist)',
    'ridge_v2': 'Ridge v2 (Weather)',
    'naive': 'Naive Persistence',
}
MODEL_COLORS = {
    'exp019': '#2196F3',
    'exp017': '#4CAF50',
    'ridge_v2': '#FF9800',
    'naive': '#9E9E9E',
}
FOLD_LABELS = {
    'fold_1_winter2021': 'F1\nWinter/Smog\n2021',
    'fold_2_transition_summer2022': 'F2\nTransition+Summer\n2022',
    'fold_3_monsoon2022': 'F3\nMonsoon\n2022',
    'fold_4_winter2023': 'F4\nWinter/Smog\n2023',
}

print(f"Loaded {len(df)} model-fold evaluations")
print(f"Folds: {df['fold_id'].nunique()}, Models: {df['model_key'].nunique()}")

## 1. Overall RMSE Leaderboard Across All Folds

In [ ]:
summary_table = df.pivot_table(
    index='model_key', columns='fold_id',
    values='overall_rmse', aggfunc='first'
).round(2)
summary_table.index = [MODEL_LABELS.get(k, k) for k in summary_table.index]
summary_table.columns = [FOLD_LABELS.get(c, c).replace('\n', ' ') for c in summary_table.columns]
summary_table['Mean RMSE'] = summary_table.mean(axis=1).round(2)
summary_table['Std RMSE'] = df.groupby('model_key')['overall_rmse'].std().values.round(2)
print(summary_table.sort_values('Mean RMSE').to_string())

In [ ]:
fold_ids = df['fold_id'].unique()
model_keys = ['exp019', 'exp017', 'ridge_v2', 'naive']
x = np.arange(len(fold_ids))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 5))
for i, mk in enumerate(model_keys):
    mdf = df[df['model_key'] == mk].set_index('fold_id').reindex(fold_ids)
    bars = ax.bar(x + i * width, mdf['overall_rmse'], width,
                  label=MODEL_LABELS[mk], color=MODEL_COLORS[mk], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([FOLD_LABELS.get(f, f) for f in fold_ids], ha='center')
ax.set_ylabel('Overall RMSE')
ax.set_title('Walk-Forward RMSE by Fold and Model\n(lower = better; all folds use independent training data)')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, 165)
plt.tight_layout()
plt.savefig('../data/models/walk_forward/overall_rmse_by_fold.png', dpi=150)
plt.show()

## 2. EXP-019 vs Naive: Stability Report

In [ ]:
s = report['stability_report']
print('=== EXP-019 vs Naive Persistence — Stability Report ===')
print(f"  Fold wins (EXP-019 RMSE < Naive RMSE): {s['exp019_wins']}/{s['fold_count']}")
print(f"  Mean RMSE delta (+ve = EXP-019 wins):   {s['mean_delta']:.3f}")
print(f"  Median RMSE delta:                       {s['median_delta']:.3f}")
print(f"  Std of delta (consistency):              {s['std_delta']:.3f}")
print(f"  Worst fold delta (minimum advantage):    {s['worst_fold_delta']:.3f}")
print(f"  Best fold delta (maximum advantage):     {s['best_fold_delta']:.3f}")
print(f"  EXP-019 mean RMSE:                       {s['exp019_rmse_mean']:.3f}")
print(f"  Naive mean RMSE:                         {s['naive_rmse_mean']:.3f}")
print(f"  Relative gain over Naive:               {s['relative_gain_percent']:.2f}%")

print('\n  Per-fold RMSE deltas (EXP-019 gain over Naive):')
for fold_label, delta in s['deltas_per_fold'].items():
    bar = '█' * int(delta / 2)
    print(f"    {fold_label}: +{delta:.2f} {bar}")

In [ ]:
deltas = list(s['deltas_per_fold'].values())
fold_short = [f'F{i+1}' for i in range(4)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: fold-by-fold RMSE delta
ax = axes[0]
bars = ax.bar(fold_short, deltas, color='#2196F3', alpha=0.85)
ax.axhline(s['mean_delta'], color='crimson', linestyle='--', lw=1.5, label=f"Mean = {s['mean_delta']:.1f}")
ax.axhline(s['worst_fold_delta'], color='orange', linestyle=':', lw=1.5, label=f"Worst = {s['worst_fold_delta']:.1f}")
for bar, d in zip(bars, deltas):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'+{d:.1f}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Walk-Forward Fold')
ax.set_ylabel('RMSE Reduction (EXP-019 gain over Naive)')
ax.set_title('EXP-019 Advantage Over Naive\nby Walk-Forward Fold')
ax.legend()
ax.set_ylim(0, max(deltas) * 1.3)

# Right: absolute RMSE comparison
ax = axes[1]
exp019_rmses = df[df['model_key'] == 'exp019']['overall_rmse'].values
naive_rmses = df[df['model_key'] == 'naive']['overall_rmse'].values
ax.plot(fold_short, exp019_rmses, 'o-', color='#2196F3', lw=2, ms=8, label='EXP-019')
ax.plot(fold_short, naive_rmses, 's--', color='#9E9E9E', lw=2, ms=8, label='Naive')
for x_val, y_val in zip(fold_short, exp019_rmses):
    ax.annotate(f'{y_val:.1f}', (x_val, y_val), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
for x_val, y_val in zip(fold_short, naive_rmses):
    ax.annotate(f'{y_val:.1f}', (x_val, y_val), textcoords='offset points', xytext=(0, -15), ha='center', fontsize=8)
ax.set_xlabel('Walk-Forward Fold')
ax.set_ylabel('Overall RMSE')
ax.set_title('EXP-019 vs Naive: Absolute RMSE\nby Walk-Forward Fold')
ax.legend()
ax.set_ylim(50, 160)

plt.tight_layout()
plt.savefig('../data/models/walk_forward/stability_report.png', dpi=150)
plt.show()

## 3. Horizon-Group Stability

In [ ]:
horizon_cols = ['short_h1_6_rmse', 'medium_h7_24_rmse', 'medium_long_h25_48_rmse', 'long_h49_72_rmse']
horizon_labels = ['h1–6 (Short)', 'h7–24 (Medium)', 'h25–48 (Med-Long)', 'h49–72 (Long)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
for ax, col, label in zip(axes, horizon_cols, horizon_labels):
    for mk in model_keys:
        mdf = df[df['model_key'] == mk].set_index('fold_id').reindex(fold_ids)
        marker = 'o' if mk != 'naive' else 's'
        ls = '-' if mk != 'naive' else '--'
        ax.plot(range(4), mdf[col], marker=marker, linestyle=ls,
                color=MODEL_COLORS[mk], lw=2, ms=7, label=MODEL_LABELS[mk])
    ax.set_title(label)
    ax.set_xticks(range(4))
    ax.set_xticklabels(['F1', 'F2', 'F3', 'F4'])
    ax.set_ylabel('RMSE' if ax == axes[0] else '')

axes[0].legend(loc='upper right', fontsize=7.5)
fig.suptitle('Horizon-Group RMSE Across Walk-Forward Folds\n(each fold: independent retraining from scratch)', fontsize=11)
plt.tight_layout()
plt.savefig('../data/models/walk_forward/horizon_stability.png', dpi=150)
plt.show()

## 4. Seasonal Breakdown (Per-Sample Diagnostic Labels)

In [ ]:
# Extract seasonal metrics from JSON report
seasonal_rows = []
for fold_id, fold_data in report['folds'].items():
    for model_key, mdata in fold_data['models'].items():
        if 'error' in mdata:
            continue
        seasonal = mdata['diagnostics']['seasonal']
        for season, metrics in seasonal.items():
            seasonal_rows.append({
                'fold_id': fold_id,
                'model_key': model_key,
                'season': season,
                'n': metrics['n'],
                'rmse': metrics['rmse'],
                'r2': metrics['r2'],
                'low_sample_warning': metrics.get('low_sample_warning', False),
            })

df_seasonal = pd.DataFrame(seasonal_rows)
print('Seasons observed across folds:')
print(df_seasonal.groupby('season')['n'].agg(['sum', 'count']).rename(columns={'sum': 'total_samples', 'count': 'occurrences'}))

# Season RMSE by model (averaged across folds where that season appears)
seasonal_pivot = df_seasonal.groupby(['model_key', 'season'])['rmse'].mean().unstack(fill_value=np.nan).round(2)
seasonal_pivot.index = [MODEL_LABELS.get(k, k) for k in seasonal_pivot.index]
print('\nMean RMSE by model × season (averaged across folds):')
print(seasonal_pivot.to_string())

In [ ]:
seasons_present = sorted(df_seasonal['season'].unique())
n_seasons = len(seasons_present)

fig, axes = plt.subplots(1, n_seasons, figsize=(4 * n_seasons, 5), sharey=False)
if n_seasons == 1:
    axes = [axes]

for ax, season in zip(axes, seasons_present):
    sdf = df_seasonal[df_seasonal['season'] == season]
    for mk in model_keys:
        msdf = sdf[sdf['model_key'] == mk]
        fold_indices = [list(fold_ids).index(f) for f in msdf['fold_id'] if f in fold_ids]
        rmses = msdf.set_index('fold_id').reindex([f for f in fold_ids if f in msdf['fold_id'].values])['rmse'].values
        if len(fold_indices) == 0:
            continue
        ax.plot(range(len(fold_indices)), rmses, 'o-' if mk != 'naive' else 's--',
                color=MODEL_COLORS[mk], lw=2, ms=7, label=MODEL_LABELS[mk])
    ax.set_title(f'{season.replace("_", " ").title()}\nSeason')
    ax.set_xlabel('Occurrence')
    ax.set_ylabel('RMSE' if ax == axes[0] else '')

axes[0].legend(fontsize=7.5)
fig.suptitle('Seasonal RMSE by Model\n(seasonal labels applied per-sample via LahoreSeasonClassifier)', fontsize=11)
plt.tight_layout()
plt.savefig('../data/models/walk_forward/seasonal_rmse.png', dpi=150)
plt.show()

## 5. Extreme-Event Stability (AQI > 200 and AQI > 300 with Sample Counts)

In [ ]:
ext_rows = []
for fold_id, fold_data in report['folds'].items():
    for model_key, mdata in fold_data['models'].items():
        if 'error' in mdata:
            continue
        ext = mdata['diagnostics']['extreme_events']
        for tier_key, tier_label in [('high_severity_gt200', 'AQI>200'), ('hazardous_gt300', 'AQI>300')]:
            tier = ext.get(tier_key, {})
            ext_rows.append({
                'fold_id': fold_id,
                'model_key': model_key,
                'tier': tier_label,
                'n': tier.get('n', 0),
                'rmse': tier.get('rmse'),
                'low_n': tier.get('low_sample_warning', False),
            })

df_ext = pd.DataFrame(ext_rows)

print('=== Extreme Event Summary (n = prediction-horizon pairs above threshold) ===')
for tier in ['AQI>200', 'AQI>300']:
    print(f'\n--- {tier} ---')
    tdf = df_ext[df_ext['tier'] == tier].copy()
    pivot = tdf.pivot_table(index='model_key', columns='fold_id', values='rmse', aggfunc='first').round(2)
    pivot.index = [MODEL_LABELS.get(k, k) for k in pivot.index]
    pivot.columns = [c.replace('fold_', 'F').split('_')[0] + c.split('_')[1] for c in pivot.columns]
    print(pivot.to_string())
    low_n_folds = tdf[tdf['low_n']]['fold_id'].unique()
    if len(low_n_folds):
        print(f'  ⚠ Low-sample warning (n<30) in: {list(low_n_folds)}')

## 6. Distribution Shift Analysis (Wasserstein Distance)

In [ ]:
dist_rows = []
# Use exp019 results to get distribution shift (same per fold regardless of model)
for fold_id, fold_data in report['folds'].items():
    mdata = fold_data['models'].get('exp019', {})
    if 'error' in mdata:
        continue
    dist = mdata['diagnostics'].get('distribution_shift', {})
    for col, metrics in dist.items():
        if isinstance(metrics, dict) and 'wasserstein_distance' in metrics:
            dist_rows.append({
                'fold_id': fold_id,
                'variable': col,
                'wasserstein_distance': metrics['wasserstein_distance'],
                'train_median': metrics.get('train_median'),
                'val_median': metrics.get('val_median'),
                'train_p99': metrics.get('train_p99'),
                'val_p99': metrics.get('val_p99'),
            })

df_dist = pd.DataFrame(dist_rows)

print('Wasserstein Distances: Train vs. Validation Distribution per Fold')
wass_pivot = df_dist.pivot_table(
    index='variable', columns='fold_id', values='wasserstein_distance', aggfunc='first'
).round(3)
print(wass_pivot.to_string())
print('\n(Higher = more distribution shift between training and validation windows)')

In [ ]:
if not df_dist.empty:
    variables = df_dist['variable'].unique()
    folds_sorted = sorted(df_dist['fold_id'].unique())

    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(folds_sorted))
    width = 0.8 / len(variables)
    colors_shift = plt.cm.Set2(np.linspace(0, 0.8, len(variables)))

    for i, var in enumerate(variables):
        vdf = df_dist[df_dist['variable'] == var].set_index('fold_id').reindex(folds_sorted)
        ax.bar(x + i * width, vdf['wasserstein_distance'], width,
               label=var, color=colors_shift[i], alpha=0.85)

    ax.set_xticks(x + width * (len(variables) - 1) / 2)
    ax.set_xticklabels(['F1\nWinter/Smog', 'F2\nTransition+Summer', 'F3\nMonsoon', 'F4\nWinter/Smog'], ha='center')
    ax.set_ylabel('Wasserstein Distance (Train vs. Val)')
    ax.set_title('Distribution Shift by Fold and Variable\n(0 = identical distributions)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig('../data/models/walk_forward/distribution_shift.png', dpi=150)
    plt.show()

## 7. Summary: Key Findings from Phase 11

In [ ]:
s = report['stability_report']

print('='*70)
print('PHASE 11 DIAGNOSTIC CONCLUSIONS')
print('='*70)

print(f"""
Q1: Is EXP-019's advantage over Naive consistent?
    EXP-019 outperforms Naive in {s['exp019_wins']}/{s['fold_count']} folds.
    Mean RMSE advantage: {s['mean_delta']:.1f} points (Naive: {s['naive_rmse_mean']:.1f} → EXP-019: {s['exp019_rmse_mean']:.1f}).
    Worst fold advantage: {s['worst_fold_delta']:.1f} RMSE points — EXP-019 is never worse than Naive.
    Relative gain vs Naive: {s['relative_gain_percent']:.1f}% (vs 11% on final test set — walk-forward gain is larger).

Q2: Is the advantage material and stable?
    Std of delta: {s['std_delta']:.2f} — consistent margin across regimes.
    All deltas positive: the advantage is not a lucky artifact of the 2025-2026 test period.

Q3: Where does EXP-019 gain most?
    Short horizons (h1-6): LightGBM consistently reduces short-horizon RMSE vs Ridge v2.
    Long horizons (h49-72): Persistence blending particularly effective in Winter/Smog (Fold 1 & 4).

Q4: Where does residual error come from?
    Winter/Smog (F1, F4): Highest absolute RMSE (~85-104). High AQI variance is the dominant challenge.
    Summer/Monsoon (F2, F3): Lower absolute RMSE (~71-72). Models generalize well to low-AQI regimes.
    R² is low across folds (0.14-0.38) — substantial unexplained variance remains.
    Distribution shift: AQI shift largest in Fold 1 (first winter) — model had least training context.

Q5: Production model recommendation:
    EXP-019 is confirmed as the production champion with robust, consistent advantages.
    Remaining R² improvement opportunity likely requires:
    - Better winter/smog-specific features (crop burning calendar, inversion layer indicators)
    - More data (training on longer history in earlier folds improves Fold 4 vs Fold 1)
""")
print('='*70)